In [76]:
import pandas as pd
import numpy as np

#### Loading csv 

In [77]:
df = pd.read_csv(r"C:\Users\LakshanyaRK\Desktop\ds_mini\cc_calls.csv")
print(f"Raw shape : {df.shape}")


Raw shape : (32882, 33)


### Null value analysis

In [78]:
null_summary = df.isnull().sum()
null_summary.sort_values(ascending=False)

Co_Ref                                      1196
cc_issues_within_questionnaire               466
cc_care_package                              138
cc_care_package_discussed                    138
cc_urgency_getting_on_site                   138
cc_external_consultant                       138
cc_agent_cross_sell_attempt                  138
cc_customer_issues_concerns                  138
cc_business_struggles_financial_hardship     138
cc_call_initiated_by                         138
cc_contractor_suggest_leave                   95
cc_contractor_sentiment_issues_score          95
cc_contractor_sentiment_start_score           95
cc_contractor_sentiment                       95
cc_pricing_sentiment_impact                   95
cc_contractor_complained                      95
cc_pricing_mentioned                          95
cc_contractor_sentiment_overall_score         95
cc_refund_discussed                           95
cc_contractor_sentiment_end_score             95
cc_process_complexit

In [79]:
df = df.fillna("Not Mentioned")

### Cleaning cc_care_package Column

Issues found:
- Multiple packages in one cell
- "Not discussed" with extra description
- Mixed values like "yes/no"
- Blank entries

Fixes:
- Replace multiple entries → "Multiple Packages"
- Standardize "Not Discussed"
- Replace "yes/no" → "Not Mentioned"

In [80]:
col = 'cc_care_package'

df[col] = df[col].astype(str).str.lower().str.strip()

# Replace blanks
df[col] = df[col].replace(['', 'nan', 'none'], 'not mentioned')

# Multiple packages
df[col] = df[col].apply(lambda x: 'multiple packages' if ',' in x else x)

# Standardize "not discussed"
df[col] = df[col].apply(lambda x: 'not discussed' if 'not discussed' in x else x)

# Handle yes/no in same cell
df[col] = df[col].apply(lambda x: 'not mentioned' if ('yes' in x and 'no' in x) else x)

#### Remove duplicates

In [81]:
before = len(df)
df = df.drop_duplicates()
print(f"Dropped {before - len(df)} duplicate rows  →  {len(df)} rows remaining")

Dropped 93 duplicate rows  →  32789 rows remaining


#### call date to date time

In [82]:
df["Call_Date"] = pd.to_datetime(df["Call_Date"], format="%d-%m-%Y", errors="coerce")

####  Cast Contact_ID to nullable Int64

In [83]:
df["Contact_ID"] = df["Contact_ID"].astype("Int64")

#### Standardise binary Yes/No columns

In [84]:
BINARY_COLS = [
    "cc_care_package_discussed",
    "cc_urgency_getting_on_site",
    "cc_external_consultant",
    "cc_agent_cross_sell_attempt",
    "cc_customer_issues_concerns",
    "cc_business_struggles_financial_hardship",
    "cc_questionnaire_completion",
    "cc_chasing_response",
    "cc_issues_within_questionnaire",
    "cc_login_issues",
    "cc_platform_issues",
    "cc_dissatisfaction_time_to_complete",
    "cc_process_complexity_concerns",
    "cc_questions_harder_than_expected",
    "cc_dissatisfaction_support",
    "cc_pricing_mentioned",
    "cc_pricing_sentiment_impact",
    "cc_refund_discussed",
    "cc_contractor_suggest_leave",
    "cc_contractor_complained",
]
 
VALID_BINARY = {"yes", "no", "not applicable", "not mentioned"}

# Normalisation map applied before the validity check

In [85]:
NORMALISE_BINARY = {
    "not applicable to this scenario": "Not Applicable",
    "not mentioned": "Not Applicable",
}
 
def clean_binary(series: pd.Series) -> pd.Series:
    s = series.str.strip()
    # Apply explicit normalisation first
    s = s.replace(NORMALISE_BINARY)
    # Title-case for consistency
    s = s.str.title()
    # Anything not in the valid set → NaN
    mask = s.str.lower().isin(VALID_BINARY)
    s = s.where(mask, other=np.nan)
    return s
 
for col in BINARY_COLS:
    df[col] = clean_binary(df[col])

#### Standardise cc_call_initiated_by

In [86]:
VALID_INITIATED = {"agent", "customer", "not relevant", "not applicable"}
 
NORMALISE_INITIATED = {
    "no": np.nan,   # 'No' is not a valid initiator
}
 
def clean_initiated(series: pd.Series) -> pd.Series:
    s = series.str.strip().str.lower()
    s = s.replace(NORMALISE_INITIATED)
    # Title-case, then invalidate anything outside the valid set
    s_title = s.str.title()
    mask = s.isin(VALID_INITIATED)
    return s_title.where(mask, other=np.nan)
 
df["cc_call_initiated_by"] = clean_initiated(df["cc_call_initiated_by"])

#### Standardise cc_contractor_sentiment

In [87]:
VALID_SENTIMENT = {"dissatisfied", "neutral", "satisfied"}
 
def clean_sentiment(series: pd.Series) -> pd.Series:
    s = series.str.strip().str.lower()
    mask = s.isin(VALID_SENTIMENT)
    return s.str.title().where(mask, other=np.nan)
 
df["cc_contractor_sentiment"] = clean_sentiment(df["cc_contractor_sentiment"])

#### Convert sentiment score columns to numeric

In [88]:
SCORE_COLS = [
    "cc_contractor_sentiment_start_score",
    "cc_contractor_sentiment_end_score",
    "cc_contractor_sentiment_overall_score",
    "cc_contractor_sentiment_issues_score",
]
 
for col in SCORE_COLS:
    df[col] = pd.to_numeric(df[col], errors="coerce")

#### FINAL REPORT

In [89]:
print(f"\nCleaned shape : {df.shape}")
print("\nNull counts per column:")
null_counts = df.isnull().sum()
print(null_counts.to_string())


Cleaned shape : (32789, 33)

Null counts per column:
Contact_ID                                      0
Call_Date                                       0
Direction                                       0
cc_care_package                                 0
cc_care_package_discussed                      23
cc_urgency_getting_on_site                     23
cc_external_consultant                         23
cc_agent_cross_sell_attempt                    43
cc_customer_issues_concerns                    28
cc_business_struggles_financial_hardship       25
cc_call_initiated_by                          182
cc_questionnaire_completion                    11
cc_chasing_response                           193
cc_issues_within_questionnaire                118
cc_login_issues                                20
cc_platform_issues                             12
cc_dissatisfaction_time_to_complete            11
cc_process_complexity_concerns                 33
cc_questions_harder_than_expected             

In [90]:
df.duplicated().any()



np.False_

In [91]:
df

,Contact_ID,Call_Date,Direction,cc_care_package,cc_care_package_discussed,cc_urgency_getting_on_site,cc_external_consultant,cc_agent_cross_sell_attempt,cc_customer_issues_concerns,cc_business_struggles_financial_hardship,...,cc_contractor_sentiment_overall_score,cc_contractor_sentiment_issues_score,cc_pricing_mentioned,cc_pricing_sentiment_impact,cc_refund_discussed,cc_contractor_suggest_leave,cc_contractor_complained,Co_Ref,Analysed_Call,Call_Year
0,625513000000,2025-05-08,OUT_BOUND,standard,Yes,No,No,No,Yes,Yes,...,30.0,20.0,Yes,Yes,No,Yes,Yes,HV3323,1,2025
1,591087000000,2024-11-25,OUT_BOUND,standard,Yes,No,No,No,Yes,No,...,0.0,0.0,Yes,Yes,No,Yes,Yes,PJ7066,1,2024
2,565091000000,2024-10-23,IN_BOUND,standard,Yes,No,No,No,Yes,No,...,40.0,20.0,Yes,Yes,No,Yes,Yes,DP6030,1,2024
3,593975000000,2025-01-13,IN_BOUND,premier,Yes,No,No,No,Yes,Yes,...,40.0,30.0,Yes,Yes,Yes,Yes,Yes,AM2413,1,2025
4,622282000000,2025-03-19,IN_BOUND,standard,Yes,No,No,No,Yes,Yes,...,40.0,20.0,Yes,Yes,No,Yes,Yes,ED6707,1,2025
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32877,596362000000,2025-02-17,OUT_BOUND,not discussed,No,No,No,No,No,No,...,NaN,NaN,No,No,No,No,No,ZP3956,1,2025
32878,623633000000,2025-04-08,IN_BOUND,not discussed,No,No,No,No,No,No,...,90.0,90.0,No,No,No,No,No,WH1545,1,2025
32879,677234000000,2025-07-01,IN_BOUND,not discussed,No,Yes,No,No,Yes,No,...,85.0,80.0,Yes,Yes,No,No,No,MK2093,1,2025
32880,592898000000,2024-12-23,OUT_BOUND,not discussed,No,No,No,No,No,No,...,0.0,NaN,No,No,No,Yes,Yes,TP1762,1,2024


In [92]:
null_counts = df.isnull().sum()
print(null_counts[null_counts > 0])

cc_care_package_discussed                      23
cc_urgency_getting_on_site                     23
cc_external_consultant                         23
cc_agent_cross_sell_attempt                    43
cc_customer_issues_concerns                    28
cc_business_struggles_financial_hardship       25
cc_call_initiated_by                          182
cc_questionnaire_completion                    11
cc_chasing_response                           193
cc_issues_within_questionnaire                118
cc_login_issues                                20
cc_platform_issues                             12
cc_dissatisfaction_time_to_complete            11
cc_process_complexity_concerns                 33
cc_questions_harder_than_expected              14
cc_dissatisfaction_support                     12
cc_contractor_sentiment                      2555
cc_contractor_sentiment_start_score           244
cc_contractor_sentiment_end_score             243
cc_contractor_sentiment_overall_score        4935


In [93]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 32789 entries, 0 to 32881
Data columns (total 33 columns):
 #   Column                                    Non-Null Count  Dtype         
---  ------                                    --------------  -----         
 0   Contact_ID                                32789 non-null  Int64         
 1   Call_Date                                 32789 non-null  datetime64[ns]
 2   Direction                                 32789 non-null  object        
 3   cc_care_package                           32789 non-null  object        
 4   cc_care_package_discussed                 32766 non-null  object        
 5   cc_urgency_getting_on_site                32766 non-null  object        
 6   cc_external_consultant                    32766 non-null  object        
 7   cc_agent_cross_sell_attempt               32746 non-null  object        
 8   cc_customer_issues_concerns               32761 non-null  object        
 9   cc_business_struggles_financial_h

#### Converted categorical columns to 'category' dtype to:
- Reduce memory usage
- Improve performance
- Optimize dataset for modeling

In [94]:
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].astype('category')

In [95]:
!cd

c:\Users\LakshanyaRK\Desktop\datascience-mp\Notebook


In [96]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 32789 entries, 0 to 32881
Data columns (total 33 columns):
 #   Column                                    Non-Null Count  Dtype         
---  ------                                    --------------  -----         
 0   Contact_ID                                32789 non-null  Int64         
 1   Call_Date                                 32789 non-null  datetime64[ns]
 2   Direction                                 32789 non-null  category      
 3   cc_care_package                           32789 non-null  category      
 4   cc_care_package_discussed                 32766 non-null  category      
 5   cc_urgency_getting_on_site                32766 non-null  category      
 6   cc_external_consultant                    32766 non-null  category      
 7   cc_agent_cross_sell_attempt               32746 non-null  category      
 8   cc_customer_issues_concerns               32761 non-null  category      
 9   cc_business_struggles_financial_h

In [97]:
(df == "").sum()

Contact_ID                                  0
Call_Date                                   0
Direction                                   0
cc_care_package                             0
cc_care_package_discussed                   0
cc_urgency_getting_on_site                  0
cc_external_consultant                      0
cc_agent_cross_sell_attempt                 0
cc_customer_issues_concerns                 0
cc_business_struggles_financial_hardship    0
cc_call_initiated_by                        0
cc_questionnaire_completion                 0
cc_chasing_response                         0
cc_issues_within_questionnaire              0
cc_login_issues                             0
cc_platform_issues                          0
cc_dissatisfaction_time_to_complete         0
cc_process_complexity_concerns              0
cc_questions_harder_than_expected           0
cc_dissatisfaction_support                  0
cc_contractor_sentiment                     0
cc_contractor_sentiment_start_scor

In [98]:
(df.apply(lambda x: x.astype(str).str.strip() == "")).sum()

Contact_ID                                  0
Call_Date                                   0
Direction                                   0
cc_care_package                             0
cc_care_package_discussed                   0
cc_urgency_getting_on_site                  0
cc_external_consultant                      0
cc_agent_cross_sell_attempt                 0
cc_customer_issues_concerns                 0
cc_business_struggles_financial_hardship    0
cc_call_initiated_by                        0
cc_questionnaire_completion                 0
cc_chasing_response                         0
cc_issues_within_questionnaire              0
cc_login_issues                             0
cc_platform_issues                          0
cc_dissatisfaction_time_to_complete         0
cc_process_complexity_concerns              0
cc_questions_harder_than_expected           0
cc_dissatisfaction_support                  0
cc_contractor_sentiment                     0
cc_contractor_sentiment_start_scor

In [99]:
for col in df.columns:
    blanks = df[col].astype(str).str.strip().eq("").sum()
    if blanks > 0:
        print(f"{col}: {blanks}")

In [100]:
(df.apply(lambda x: x.astype(str).str.strip() == "")).sum()

Contact_ID                                  0
Call_Date                                   0
Direction                                   0
cc_care_package                             0
cc_care_package_discussed                   0
cc_urgency_getting_on_site                  0
cc_external_consultant                      0
cc_agent_cross_sell_attempt                 0
cc_customer_issues_concerns                 0
cc_business_struggles_financial_hardship    0
cc_call_initiated_by                        0
cc_questionnaire_completion                 0
cc_chasing_response                         0
cc_issues_within_questionnaire              0
cc_login_issues                             0
cc_platform_issues                          0
cc_dissatisfaction_time_to_complete         0
cc_process_complexity_concerns              0
cc_questions_harder_than_expected           0
cc_dissatisfaction_support                  0
cc_contractor_sentiment                     0
cc_contractor_sentiment_start_scor

In [101]:
# Calculate null counts
null_counts = df.isnull().sum()

# Calculate missing percentage
missing_percentage = (df.isnull().sum() / len(df)) * 100

# Combine into a report
missing_report = pd.DataFrame({
    'Null_Count': null_counts,
    'Missing_Percentage': missing_percentage.round(2)
})

print("\n===== Missing Value Report =====")
print(missing_report)

# Optional: Save report to CSV
missing_report.to_csv("missing_report.csv", index=True)


===== Missing Value Report =====
                                          Null_Count  Missing_Percentage
Contact_ID                                         0                0.00
Call_Date                                          0                0.00
Direction                                          0                0.00
cc_care_package                                    0                0.00
cc_care_package_discussed                         23                0.07
cc_urgency_getting_on_site                        23                0.07
cc_external_consultant                            23                0.07
cc_agent_cross_sell_attempt                       43                0.13
cc_customer_issues_concerns                       28                0.09
cc_business_struggles_financial_hardship          25                0.08
cc_call_initiated_by                             182                0.56
cc_questionnaire_completion                       11                0.03
cc_chasing_respon

In [102]:

# Check number of duplicate rows
duplicate_count = df.duplicated().sum()
print(f"Total duplicate rows: {duplicate_count}")

# Show duplicate rows (if any)
duplicate_rows = df[df.duplicated()]
print("\n===== Duplicate Rows =====")
print(duplicate_rows)



Total duplicate rows: 0

===== Duplicate Rows =====
Empty DataFrame
Columns: [Contact_ID, Call_Date, Direction, cc_care_package, cc_care_package_discussed, cc_urgency_getting_on_site, cc_external_consultant, cc_agent_cross_sell_attempt, cc_customer_issues_concerns, cc_business_struggles_financial_hardship, cc_call_initiated_by, cc_questionnaire_completion, cc_chasing_response, cc_issues_within_questionnaire, cc_login_issues, cc_platform_issues, cc_dissatisfaction_time_to_complete, cc_process_complexity_concerns, cc_questions_harder_than_expected, cc_dissatisfaction_support, cc_contractor_sentiment, cc_contractor_sentiment_start_score, cc_contractor_sentiment_end_score, cc_contractor_sentiment_overall_score, cc_contractor_sentiment_issues_score, cc_pricing_mentioned, cc_pricing_sentiment_impact, cc_refund_discussed, cc_contractor_suggest_leave, cc_contractor_complained, Co_Ref, Analysed_Call, Call_Year]
Index: []

[0 rows x 33 columns]


In [103]:
df.cc_contractor_sentiment_end_score.info()

<class 'pandas.core.series.Series'>
Index: 32789 entries, 0 to 32881
Series name: cc_contractor_sentiment_end_score
Non-Null Count  Dtype  
--------------  -----  
32546 non-null  float64
dtypes: float64(1)
memory usage: 512.3 KB


In [104]:

# Replace blanks ("" or spaces) with "Not Mentioned"
df['cc_contractor_sentiment'] = df['cc_contractor_sentiment'].astype(str).str.strip()
df['cc_contractor_sentiment'] = df['cc_contractor_sentiment'].replace("", "Not Mentioned")



### Use Median to Fill Missing Sentiment Scores
The sentiment score columns contain numeric values such as 0, 20, 40, 50, 80, and 90.
Since 0 is a valid score, missing values should not be replaced with 0.
Instead, the median is the safest choice because:

The dataset has skewed values, and the median is not affected by extreme highs or lows.
Median preserves the natural distribution of the sentiment scores.
It avoids incorrectly treating missing values as real low scores (0).

Using the median ensures clean, consistent, and unbiased data for analysis or modeling.

In [105]:
cols = [
    "cc_contractor_sentiment_start_score",
    "cc_contractor_sentiment_end_score",
    "cc_contractor_sentiment_overall_score",
    "cc_contractor_sentiment_issues_score"
]

# Convert to string & strip whitespace
df[cols] = df[cols].astype(str).apply(lambda col: col.str.strip())

# Replace blanks & placeholder text with NaN
df[cols] = df[cols].replace(
    ["", " ", "Not Applicable", "not applicable", "NA", "N/A"],
    np.nan
)

# Convert columns to numeric (very important)
for col in cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Fill NaN with median of each column
for col in cols:
    median_value = df[col].median()
    df[col].fillna(median_value, inplace=True)


C:\Users\LakshanyaRK\AppData\Local\Temp\ipykernel_22420\2428932033.py:24: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(median_value, inplace=True)


In [106]:
# Columns to fill with MODE
mode_cols = [
    "cc_pricing_mentioned",
    "cc_pricing_sentiment_impact",
    "cc_refund_discussed",
    "cc_contractor_suggest_leave",
    "cc_contractor_complained"
]

# Step 1: Convert to string and strip spaces
df[mode_cols] = df[mode_cols].astype(str).apply(lambda col: col.str.strip())

# Step 2: Replace blanks + placeholders with NaN
df[mode_cols] = df[mode_cols].replace(
    ["", " ", "Not Applicable", "not applicable", "NA", "N/A"],
    np.nan
)

# Step 3: Fill NaN with MODE (most frequent value)
for col in mode_cols:
    mode_value = df[col].mode()[0]   # get mode
    df[col].fillna(mode_value, inplace=True)

# Save cleaned file
df.to_csv("cc_calls_dataset_mode_filled.csv", index=False)


C:\Users\LakshanyaRK\AppData\Local\Temp\ipykernel_22420\2546977253.py:22: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(mode_value, inplace=True)


In [107]:

# Calculate null count
null_count = df.isnull().sum()

# Calculate missing percentage
missing_percentage = (df.isnull().sum() / len(df)) * 100

# Create a combined report
null_report = pd.DataFrame({
    "Null Count": null_count,
    "Missing Percentage": missing_percentage.round(2)
})

print("\n===== Null Value Report =====")
print(null_report)



===== Null Value Report =====
                                          Null Count  Missing Percentage
Contact_ID                                         0                0.00
Call_Date                                          0                0.00
Direction                                          0                0.00
cc_care_package                                    0                0.00
cc_care_package_discussed                         23                0.07
cc_urgency_getting_on_site                        23                0.07
cc_external_consultant                            23                0.07
cc_agent_cross_sell_attempt                       43                0.13
cc_customer_issues_concerns                       28                0.09
cc_business_struggles_financial_hardship          25                0.08
cc_call_initiated_by                             182                0.56
cc_questionnaire_completion                       11                0.03
cc_chasing_response 

In [108]:
df.drop_duplicates(subset=['Co_Ref'], inplace=True)

In [109]:
df.to_csv("cc_calls_cleaned.csv", index=False)

In [110]:
df[cols].isna().sum()

cc_contractor_sentiment_start_score      0
cc_contractor_sentiment_end_score        0
cc_contractor_sentiment_overall_score    0
cc_contractor_sentiment_issues_score     0
dtype: int64